In [1]:
!pip install -q faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 44.5 MB/s eta 0:00:0000:0100:01


In [3]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


### Zero Shot Classifier for Q1, Q2 and Q6

In [5]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), 
              str(row_150['B']), 
              str(row_150['C']), 
              str(row_150['D']), 
              str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [11]:
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

print(result)

gt_index = result['labels'].index(ans_150)

print(f"Ground Truth option is stored in the index : {gt_index}")

gt_score = result['scores'][gt_index]

print(f"Predicted probability score assigned to the \
ground-truth correct option : {round(gt_score, 3)}")

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [34]:
# Convert row 150 prompt into embedding
query_embedding = model.encode([prompt_150])

# Search top 10 nearest documents
distances, indices = index.search(
    query_embedding,
    k=10
)

retrieved_indices = list(indices[0])

if 150 in retrieved_ids:
    rank = retrieved_ids.index(150) + 1
    print(f"Exact rank (1 through 10) the FAISS placed the true correct document : {rank}")
else:
    print("Not found in top 10")

Exact rank (1 through 10) the FAISS placed the true correct document : 10


In [50]:
for id in retrieved_ids:
    #print(id)
    print(train[train['id']==id]['prompt'])

662    Pick the best possible answer: What is the rel...
Name: prompt, dtype: object
1700    Determine the correct option: What is the form...
Name: prompt, dtype: object
1268    Identify the correct statement: What is the si...
Name: prompt, dtype: object
1531    Choose the correct answer: What is magnetic su...
Name: prompt, dtype: object
575    Pick the best possible answer: What is the fat...
Name: prompt, dtype: object
846    Identify the correct statement: What are amorp...
Name: prompt, dtype: object
1692    Identify the correct statement: What is the pr...
Name: prompt, dtype: object
1905    Select the most accurate option: What was Isaa...
Name: prompt, dtype: object
167    What is the Optical Signal-to-Noise Ratio (OSNR)?
Name: prompt, dtype: object
149    Which of the following is correct? What did Ne...
Name: prompt, dtype: object


### Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. 

Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

In [39]:
# Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair
print(ce_scores)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[4.6602316 4.6602316 4.7375245 4.7375245 4.68704   4.752575  4.752575
 4.752575  4.707247  4.7585106]


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [45]:
# Sort by cross encoder score descending
reranked_order = np.argsort(ce_scores)[::-1]

print(reranked_order)

retrieved_indices = np.array(retrieved_indices)

reranked_indices = retrieved_indices[reranked_order]

print(reranked_indices)

ce_rank = list(reranked_indices).index(150) + 1

print("Cross Encoder rank the true document at :", ce_rank)

[9 7 5 6 2 3 8 4 1 0]
[ 150 1906  847 1693 1269 1532  168  576 1701  663]
Cross Encoder rank the true document at : 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [51]:
# Get row 42 prompt
row_42 = train.iloc[42]

prompt_42 = str(row_42['prompt'])


# Step 1: Embed the prompt
query_embedding_42 = model.encode([prompt_42])


# Step 2: FAISS retrieve top k=5
distances, indices = index.search(
    query_embedding_42,
    k=5
)

retrieved_docs = [
    kb[i]
    for i in indices[0]
]


# Step 3: Concatenate documents
concatenated_docs = " ".join(retrieved_docs)


# Step 4: Create final RAG input
rag_text = (
    "Context: "
    + concatenated_docs
    + " Question: "
    + prompt_42
)


# Step 5: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)


# Step 6: Tokenize without truncation
tokens = tokenizer(
    rag_text,
    truncation=False
)


# Count tokens
token_count = len(tokens["input_ids"])

print(token_count)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [53]:
# Retrieve exact true document from KB
true_document = kb[150]


# Create RAG augmented prompt
rag_prompt = (
    "Context: "
    + true_document
    + " Question: "
    + prompt_150
)


# Run zero-shot classifier
rag_result = zs(
    rag_prompt,
    candidate_labels=labels_150
)

gt_index = rag_result["labels"].index(ans_150)

new_score = rag_result["scores"][gt_index]

print(rag_result)

print(f"The new predicted probability score of the \
ground-truth correct option : {round(new_score, 3)}")

{'sequence': 'Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on 

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [69]:
# Force WRONG retrieved context
wrong_document = kb[999]

print(wrong_document)
print()
print(prompt_150)
# Create adversarial RAG input
bad_rag_prompt = (
    "Context: "
    + wrong_document
    + " Question: "
    + prompt_150
)
print()
print(ans_150)
# Run zero-shot classifier
bad_result = zs(
    bad_rag_prompt,
    candidate_labels=labels_150
)

print()
print(bad_result)

correct_index = bad_result["labels"].index(ans_150)

bad_score = bad_result["scores"][correct_index]
print()
print(f"The probability of the correct option\
using Bad RAG : {round(bad_score, 3)}")

A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.

Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

{'sequence': 'Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average mo

### In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place)

In [64]:
hits = 0
total = 100

for idx in range(100):

    # Get row
    row = train.iloc[idx]

    # Question
    prompt = str(row["prompt"])

    # Correct answer text
    correct_answer = str(
        row[row["answer"]]
    )


    # Embed query
    query_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )


    # Retrieve top k=5 documents
    distances, indices = index.search(
        query_embedding,
        k=5
    )


    # Get retrieved documents
    retrieved_docs = [
        kb[i]
        for i in indices[0]
    ]


    # Check if exact answer string exists
    found = False

    for doc in retrieved_docs:
        if correct_answer in doc:
            found = True
            break


    if found:
        hits += 1


hit_rate = (hits / total) * 100

print("Hits:", hits)
print(
    "Hit Rate@5:",
    round(hit_rate, 1)
)

Hits: 73
Hit Rate@5: 73.0


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).



In [72]:
# Load reranker
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


# MAP@3 function
def apk(actual, predicted, k=3):
    
    predicted = predicted[:k]
    
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)
    
    return 0


scores = []


for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    options = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    candidate_labels = list(options.values())

    true_letter = row["answer"]


    # ----------------------
    # 1. RETRIEVE (FAISS)
    # ----------------------

    query_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    distances, indices = index.search(
        query_embedding,
        k=5
    )

    docs_5 = [
        kb[i]
        for i in indices[0]
    ]


    # ----------------------
    # 2. RERANK
    # ----------------------

    pairs = [
        [prompt, doc]
        for doc in docs_5
    ]

    ce_scores = cross_encoder.predict(
        pairs
    )

    best_idx = np.argmax(
        ce_scores
    )

    best_document = docs_5[best_idx]


    # ----------------------
    # 3. AUGMENT
    # ----------------------

    rag_prompt = (
        "Context: "
        + best_document
        + " Question: "
        + prompt
    )


    # ----------------------
    # 4. ZERO-SHOT PREDICT
    # ----------------------

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels
    )


    # ----------------------
    # 5. Convert labels → letters
    # ----------------------

    ranked_letters = []

    for label in result["labels"]:

        for letter, text in options.items():

            if label == text:
                ranked_letters.append(letter)
                break


    top3 = ranked_letters[:3]


    # ----------------------
    # 6. MAP@3
    # ----------------------

    score = apk(
        true_letter,
        top3,
        k=3
    )

    scores.append(score)


# Final score
final_map3 = np.mean(scores)

print(f"final average MAP@3 score of this state-of-the-art\
RAG pipeline across these 20 rows : {round(final_map3, 3)}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


final average MAP@3 score of this state-of-the-artRAG pipeline across these 20 rows : 0.975
